# 收益率模型——获取"阿尔法"完整教程：从预测变量到 IC 评估

## 📚 教学目标
1. 理解 **α 模型**的核心目标：预测资产的超额收益率
2. 掌握 **IC（信息系数）** 的定义、计算和相关评价指标
3. 学会用 **Fama-MacBeth 回归** 和 **投资组合排序法** 检验预测变量
4. 理解预测变量的 **六大筛选标准** 和 **信息衰减** 特征
5. 掌握 α 模型与风险模型的配合使用逻辑

**参考书**：《因子投资：方法与实践》（石川）第 7.1 节
**教学策略**：先用极小数据集让你"看见"每一步计算，再扩展到真实规模

---

## 1. 场景设定：收益率模型是什么？

### 🎯 核心问题

在量化投资中，**收益率模型（Alpha Model）** 的目标是预测股票的未来收益率，从而构建能够获得超额收益的投资组合。

> **关键区分**：本书中，"阿尔法"和"因子"指的是投资组合（如价值因子、特质波动率异象），
> 而非它们背后的变量。业界常说的"阿尔法因子"在本书术语中应称为 **收益预测变量（return predictors）**。

### 📖 书中的定义

> 收益率模型的作用是预测股票的收益率，从而获得超过基准的超额收益。
> 开发收益率模型的目标就是找到能够预测股票收益率的变量，这些变量往往是通过股票的量价或者财务数据计算的指标。

### 🔗 本节的逻辑链条

```
预测变量 z_it 的定义
    ↓
IC = corr(z_it, R_{i,t+1}) 衡量预测能力
    ↓
Fama-MacBeth 回归 → 因子收益率 λ_t
    ↓
投资组合排序法 → Long-Short Spread
    ↓
六大筛选标准 + 信息衰减 → 变量质量评估
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm

# 设置中文字体和样式
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. 微型数据集：10 只股票 × 1 个月

### 📐 IC 的定义

IC（Information Coefficient，信息系数）衡量预测变量的预测能力：

$$IC = \text{corr}(z_{i,t}, R_{i,t+1})$$

其中：
- $z_{i,t}$ = 股票 $i$ 在 $t$ 时刻的预测变量取值
- $R_{i,t+1}$ = 股票 $i$ 在 $t+1$ 时刻的收益率
- IC 取值范围 [-1, 1]，绝对值越高表示预测能力越强

> 经验标准：按日频收益率评价，IC 高于 2% 就是优秀的预测变量。

In [ ]:
# ========== 步骤 1: 构造微型数据集 ==========
N_STOCKS = 10
stock_names = [f'Stock_{i+1:02d}' for i in range(N_STOCKS)]

# 预测变量 z（例如：账面市值比 BM）
z = np.array([0.85, 1.20, 0.45, 1.50, 0.60, 0.95, 1.10, 0.30, 1.35, 0.70])

# 下一期收益率 R（真实值）
R_next = np.array([2.1, 3.5, 0.8, 4.2, 1.5, 2.8, 3.0, 0.5, 3.8, 1.9])

print("📊 微型数据集：10 只股票")
print(f"{'股票':<12} {'预测变量 z':<15} {'下期收益率 R':<15}")
print("-" * 42)
for i in range(N_STOCKS):
    print(f"{stock_names[i]:<12} {z[i]:<15.2f} {R_next[i]:<15.2f}")

In [ ]:
# ========== 步骤 2: 手算 IC ==========
print(f"\n📊 步骤 2: 手算 Pearson 相关系数 IC")
print(f"  计算公式: IC = corr(z, R) = Cov(z,R) / (Std(z) × Std(R))")

# 计算均值
z_mean = z.mean()
R_mean = R_next.mean()
print(f"\n  z 的均值 = {z_mean:.4f}")
print(f"  R 的均值 = {R_mean:.4f}")

# 计算协方差
cov_zR = np.sum((z - z_mean) * (R_next - R_mean)) / (N_STOCKS - 1)
print(f"\n  Cov(z, R) = {cov_zR:.4f}")

# 计算标准差
std_z = np.sqrt(np.sum((z - z_mean)**2) / (N_STOCKS - 1))
std_R = np.sqrt(np.sum((R_next - R_mean)**2) / (N_STOCKS - 1))
print(f"  Std(z) = {std_z:.4f}")
print(f"  Std(R) = {std_R:.4f}")

# 计算 IC
IC_manual = cov_zR / (std_z * std_R)
print(f"\n  💡 IC = {cov_zR:.4f} / ({std_z:.4f} × {std_R:.4f}) = {IC_manual:.4f}")
print(f"\n  💡 解读: IC = {IC_manual:.4f} > 0，说明预测变量 z 与下期收益正相关，")
print(f"     z 越高的股票，下期收益越高。")

In [ ]:
# ========== 用 scipy 验证 ==========
IC_scipy, p_value = stats.pearsonr(z, R_next)

print("🔬 scipy.stats.pearsonr 验证:")
print(f"  手算 IC = {IC_manual:.6f}")
print(f"  scipy IC = {IC_scipy:.6f}")
print(f"  P 值     = {p_value:.6f}")
print(f"\n  ✅ 验证通过！")

In [ ]:
# ========== 可视化：散点图 + 回归线 ==========
fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(z, R_next, s=100, c='steelblue', edgecolors='black', zorder=5)

# 添加回归线
slope, intercept = np.polyfit(z, R_next, 1)
x_line = np.linspace(z.min() - 0.1, z.max() + 0.1, 100)
ax.plot(x_line, slope * x_line + intercept, 'r--', linewidth=2, label=f'OLS: R = {slope:.2f}z + {intercept:.2f}')

# 标注股票名
for i in range(N_STOCKS):
    ax.annotate(stock_names[i], (z[i], R_next[i]), textcoords="offset points",
                xytext=(5, 5), fontsize=8)

ax.set_xlabel('Predictor z (e.g., B/M)', fontsize=12)
ax.set_ylabel('Next-Period Return R (%)', fontsize=12)
ax.set_title(f'Predictor vs Return: IC = {IC_manual:.4f}', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  散点图显示预测变量 z 与下期收益率 R 的关系。")
print(f"  回归线斜率为正 ({slope:.2f})，证实 z 具有正向预测能力。")
print(f"  IC = {IC_manual:.4f} 表示线性相关程度较高。")

---

## 3. IC 相关评价指标体系

### 📐 IC 相关指标

除了 IC 本身，业界还使用以下指标综合评价预测变量：

| 指标 | 公式 | 含义 |
|------|------|------|
| IC | $\text{corr}(z_t, R_{t+1})$ | 单期预测能力 |
| IC_mean | $\frac{1}{T}\sum_{t=1}^T IC_t$ | 平均预测能力 |
| IC_std | $\text{std}(IC_t)$ | 预测能力的稳定性 |
| IR | $IC_{mean} / IC_{std}$ | 信息率（类似 Sharpe Ratio） |
| IC > 2% 比例 | $\frac{\#(IC_t > 0.02)}{T}$ | 预测能力的持续性 |
| IC t-value | $IC_{mean} / SE(IC)$ | IC 是否显著异于零 |

In [ ]:
# ========== 步骤 3: 模拟多期 IC 序列 ==========
N_MONTHS = 60
N_STOCKS = 300

print(f"📊 模拟参数：")
print(f"  股票数量: {N_STOCKS} 只/月")
print(f"  时间跨度: {N_MONTHS} 个月")

# 存储每期的 IC
ic_series = []

for t in range(N_MONTHS):
    # 预测变量 z（如 B/M）
    z_t = np.random.lognormal(mean=0, sigma=0.5, size=N_STOCKS)
    
    # 收益率：z 有弱预测力 + 噪声
    # 真实 IC 约 0.03-0.05
    factor_effect = 0.5 * (z_t - z_t.mean()) / z_t.std()
    noise = np.random.normal(0, 1, N_STOCKS)
    R_t1 = factor_effect + noise
    
    # 计算当期 IC
    ic_t, _ = stats.pearsonr(z_t, R_t1)
    ic_series.append(ic_t)

ic_series = np.array(ic_series)

# ========== 计算 IC 评价指标 ==========
print(f"\n📊 IC 评价指标汇总")
print(f"  IC 均值 (IC_mean)   = {ic_series.mean():.4f}")
print(f"  IC 标准差 (IC_std)  = {ic_series.std():.4f}")
print(f"  信息率 (IR)         = {ic_series.mean() / ic_series.std():.4f}")
print(f"  IC > 2% 的比例      = {(ic_series > 0.02).mean():.2%}")

# IC 的 t 检验
ic_t_stat = ic_series.mean() / (ic_series.std() / np.sqrt(N_MONTHS))
ic_p_value = 2 * (1 - stats.t.cdf(abs(ic_t_stat), df=N_MONTHS-1))
print(f"  IC t-value          = {ic_t_stat:.4f}")
print(f"  IC p-value          = {ic_p_value:.6f}")

if abs(ic_t_stat) > 1.96:
    print(f"\n  💡 IC 显著异于零 (|t| > 1.96)，预测变量有效！")
else:
    print(f"\n  💡 IC 不显著 (|t| < 1.96)，预测变量可能无效。")

In [ ]:
# ========== 可视化 IC 时间序列 ==========
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 左图：IC 时间序列
ax1 = axes[0]
colors_ic = ['#2ecc71' if ic > 0 else '#e74c3c' for ic in ic_series]
ax1.bar(range(N_MONTHS), ic_series, color=colors_ic, alpha=0.7)
ax1.axhline(y=0, color='black', linewidth=0.8)
ax1.axhline(y=ic_series.mean(), color='blue', linewidth=2, linestyle='--',
            label=f'IC Mean = {ic_series.mean():.4f}')
ax1.axhline(y=0.02, color='orange', linewidth=1.5, linestyle=':',
            label='IC = 2% threshold')
ax1.set_xlabel('Month', fontsize=12)
ax1.set_ylabel('IC', fontsize=12)
ax1.set_title('IC Time Series', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# 右图：IC 分布
ax2 = axes[1]
ax2.hist(ic_series, bins=20, color='steelblue', alpha=0.7, edgecolor='black')
ax2.axvline(x=ic_series.mean(), color='red', linewidth=2, linestyle='--',
            label=f'Mean = {ic_series.mean():.4f}')
ax2.axvline(x=0.02, color='orange', linewidth=2, linestyle=':',
            label='2% threshold')
ax2.set_xlabel('IC Value', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.set_title('IC Distribution', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左图：IC 时间序列柱状图，绿色为正 IC，红色为负 IC。")
print(f"  蓝色虚线为 IC 均值，橙色虚线为 2% 经验阈值。")
print(f"  右图：IC 分布直方图，若均值显著偏离零则预测变量有效。")

---

## 4. Fama-MacBeth 回归检验预测变量

### 📐 Fama-MacBeth 回归

Fama-MacBeth 回归是检验预测变量的常用方法。在每期 $t$ 做截面回归：

$$R_{i,t} = \lambda_{0,t} + \lambda_{1,t} \cdot z_{i,t-1} + \varepsilon_{i,t}$$

其中：
- $R_{i,t}$ = 股票 $i$ 在 $t$ 期的收益率
- $z_{i,t-1}$ = 股票 $i$ 在 $t-1$ 期的预测变量（滞后避免前视偏差）
- $\lambda_{1,t}$ = $t$ 期的因子收益率

对 $\lambda_{1,t}$ 的时间序列做 T 检验，判断因子收益率是否显著异于零。

In [ ]:
# ========== 步骤 4: Fama-MacBeth 回归 ==========
N_STOCKS = 300
N_MONTHS = 60

# 存储每期的因子收益率
lambda_0_list = []  # 截距
lambda_1_list = []  # 因子收益率

for t in range(N_MONTHS):
    # 预测变量 z（滞后一期）
    z_t = np.random.lognormal(mean=0, sigma=0.5, size=N_STOCKS)
    
    # 收益率
    factor_effect = 0.5 * (z_t - z_t.mean()) / z_t.std()
    noise = np.random.normal(0, 1, N_STOCKS)
    R_t = factor_effect + noise
    
    # 截面回归: R_t = lambda_0 + lambda_1 * z_t + eps
    X = sm.add_constant(z_t)
    model = sm.OLS(R_t, X).fit()
    lambda_0_list.append(model.params[0])
    lambda_1_list.append(model.params[1])

lambda_1_array = np.array(lambda_1_list)

# ========== T 检验 ==========
print("📊 Fama-MacBeth 回归结果")
print(f"  因子收益率均值 (λ₁)  = {lambda_1_array.mean():.4f}")
print(f"  因子收益率标准差     = {lambda_1_array.std():.4f}")

t_stat_fm = lambda_1_array.mean() / (lambda_1_array.std() / np.sqrt(N_MONTHS))
p_value_fm = 2 * (1 - stats.t.cdf(abs(t_stat_fm), df=N_MONTHS-1))
print(f"  T 统计量            = {t_stat_fm:.4f}")
print(f"  P 值                = {p_value_fm:.6f}")

if abs(t_stat_fm) > 1.96:
    print(f"\n  💡 因子收益率显著 (|t| > 1.96)，预测变量有效！")
else:
    print(f"\n  💡 因子收益率不显著 (|t| < 1.96)，预测变量可能无效。")

In [ ]:
# ========== 可视化因子收益率时间序列 ==========
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 左图：因子收益率时间序列
ax1 = axes[0]
colors_lambda = ['#2ecc71' if l > 0 else '#e74c3c' for l in lambda_1_array]
ax1.bar(range(N_MONTHS), lambda_1_array, color=colors_lambda, alpha=0.7)
ax1.axhline(y=0, color='black', linewidth=0.8)
ax1.axhline(y=lambda_1_array.mean(), color='blue', linewidth=2, linestyle='--',
            label=f'Mean λ₁ = {lambda_1_array.mean():.4f}')
ax1.set_xlabel('Month', fontsize=12)
ax1.set_ylabel('Factor Return λ₁', fontsize=12)
ax1.set_title('Fama-MacBeth Factor Return Time Series', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# 右图：累积因子收益率
ax2 = axes[1]
cum_lambda = np.cumsum(lambda_1_array)
ax2.plot(range(N_MONTHS), cum_lambda, color='steelblue', linewidth=2)
ax2.fill_between(range(N_MONTHS), 0, cum_lambda, alpha=0.3, color='steelblue')
ax2.axhline(y=0, color='black', linewidth=0.8)
ax2.set_xlabel('Month', fontsize=12)
ax2.set_ylabel('Cumulative Factor Return (%)', fontsize=12)
ax2.set_title('Cumulative Alpha from Predictor', fontsize=14, fontweight='bold')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左图：每期截面回归得到的因子收益率 λ₁，绿色为正，红色为负。")
print(f"  右图：累积因子收益率，持续上升说明预测变量能稳定获取超额收益。")

---

## 5. 投资组合排序法检验

### 📐 分组检验

另一种检验预测变量的方法是投资组合排序法：

1. 每期按预测变量 $z$ 从高到低分 5 组
2. 计算每组的平均收益率
3. 构建多空组合：Spread = Q1(高z) - Q5(低z)
4. 对 Spread 做 T 检验

如果预测变量有效，高 z 组应获得更高收益，Spread 显著为正。

In [ ]:
# ========== 步骤 5: 投资组合排序法 ==========
N_STOCKS = 300
N_MONTHS = 60
N_GROUPS = 5
group_labels = ['Q1(Low)', 'Q2', 'Q3', 'Q4', 'Q5(High)']

# 存储每期各组收益和 Spread
group_returns = {f'Q{i+1}': [] for i in range(N_GROUPS)}
monthly_spreads = []

for t in range(N_MONTHS):
    z_t = np.random.lognormal(mean=0, sigma=0.5, size=N_STOCKS)
    factor_effect = 0.5 * (z_t - z_t.mean()) / z_t.std()
    noise = np.random.normal(0, 1, N_STOCKS)
    R_t = factor_effect + noise
    
    # 按 z 分 5 组
    df_t = pd.DataFrame({'z': z_t, 'R': R_t})
    df_t['group'] = pd.qcut(df_t['z'], N_GROUPS, labels=[f'Q{i+1}' for i in range(N_GROUPS)])
    
    # 计算各组平均收益
    group_avg = df_t.groupby('group')['R'].mean()
    for i in range(N_GROUPS):
        group_returns[f'Q{i+1}'].append(group_avg.iloc[i])
    
    # Spread = Q5(高z) - Q1(低z)
    spread = group_avg.iloc[-1] - group_avg.iloc[0]
    monthly_spreads.append(spread)

monthly_spreads = np.array(monthly_spreads)

# ========== T 检验 ==========
print("📊 投资组合排序法结果")
print(f"  Spread 均值  = {monthly_spreads.mean():.4f}")
print(f"  Spread 标准差 = {monthly_spreads.std():.4f}")

t_stat_sort = monthly_spreads.mean() / (monthly_spreads.std() / np.sqrt(N_MONTHS))
p_value_sort = 2 * (1 - stats.t.cdf(abs(t_stat_sort), df=N_MONTHS-1))
print(f"  T 统计量    = {t_stat_sort:.4f}")
print(f"  P 值        = {p_value_sort:.6f}")

# Sharpe Ratio
sr_monthly = monthly_spreads.mean() / monthly_spreads.std()
sr_annual = sr_monthly * np.sqrt(12)
print(f"\n  月度 Sharpe Ratio = {sr_monthly:.4f}")
print(f"  年化 Sharpe Ratio = {sr_annual:.4f}")
print(f"  📐 验证: t = SR_monthly × √T = {sr_monthly:.4f} × √{N_MONTHS} = {sr_monthly * np.sqrt(N_MONTHS):.4f}")

In [ ]:
# ========== 可视化分组收益 ==========
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 左图：各组平均收益
avg_group_rets = [np.mean(group_returns[f'Q{i+1}']) for i in range(N_GROUPS)]
colors_group = plt.cm.RdYlGn(np.linspace(0.15, 0.85, N_GROUPS))

ax1 = axes[0]
bars = ax1.bar(group_labels, avg_group_rets, color=colors_group, edgecolor='black', alpha=0.8)
ax1.axhline(y=0, color='black', linewidth=0.8)
for bar, v in zip(bars, avg_group_rets):
    va = 'bottom' if v >= 0 else 'top'
    offset = 0.02 if v >= 0 else -0.02
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + offset,
             f'{v:.3f}', ha='center', va=va, fontweight='bold')
ax1.set_xlabel('Portfolio Group (by z)', fontsize=12)
ax1.set_ylabel('Average Return', fontsize=12)
ax1.set_title('Portfolio Sort: Group Returns', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# 右图：Spread 时间序列
ax2 = axes[1]
colors_sp = ['#2ecc71' if s > 0 else '#e74c3c' for s in monthly_spreads]
ax2.bar(range(N_MONTHS), monthly_spreads, color=colors_sp, alpha=0.7)
ax2.axhline(y=0, color='black', linewidth=0.8)
ax2.axhline(y=monthly_spreads.mean(), color='blue', linewidth=2, linestyle='--',
            label=f'Mean Spread = {monthly_spreads.mean():.4f}')
ax2.set_xlabel('Month', fontsize=12)
ax2.set_ylabel('Spread (Q1 - Q5)', fontsize=12)
ax2.set_title('Long-Short Spread Time Series', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左图：按预测变量 z 分 5 组的平均收益，理想情况应单调递减。")
print(f"  右图：多空组合 Spread 时间序列，绿色为正（高z组优于低z组）。")

---

## 6. 单调性检验（Spearman 秩相关）

### 📐 单调性

除了检验 Spread 是否显著，还需检查各组收益是否**单调递减**：

$$\rho_s = \text{spearmanr}(\text{group rank}, \text{group return})$$

- $|\rho_s| > 0.8$：强单调性，因子质量优秀
- $|\rho_s| > 0.5$：中等单调性
- $|\rho_s| < 0.5$：弱单调性，因子不可靠

In [ ]:
# ========== 步骤 6: 单调性检验 ==========
group_ranks = np.arange(1, N_GROUPS + 1)
group_avg_rets = np.array([np.mean(group_returns[f'Q{i+1}']) for i in range(N_GROUPS)])

spearman_corr, spearman_p = stats.spearmanr(group_ranks, group_avg_rets)

print("📊 单调性检验结果")
print(f"  组别: {group_labels}")
print(f"  平均收益: {[f'{r:.4f}' for r in group_avg_rets]}")
print(f"  Spearman 相关系数 = {spearman_corr:.4f}")
print(f"  P 值              = {spearman_p:.6f}")

if abs(spearman_corr) > 0.8:
    print(f"\n  💡 强单调性 (|ρ| > 0.8)，预测变量质量优秀！")
elif abs(spearman_corr) > 0.5:
    print(f"\n  💡 中等单调性 (|ρ| > 0.5)，预测变量尚可。")
else:
    print(f"\n  💡 弱单调性 (|ρ| < 0.5)，预测变量不可靠。")

---

## 7. 预测变量的六大筛选标准

### 📖 书中的要点

并非所有看起来有效的预测变量都值得使用。理想的收益预测变量应满足六大标准：

| 标准 | 英文 | 含义 |
|------|------|------|
| 1. 逻辑性 | Intuitiveness | 有经济学解释（风险补偿或错误定价） |
| 2. 持续性 | Persistence | 实证上能持续预测收益 |
| 3. 信息增量性 | Information Increment | 相对于已有变量有增量信息 |
| 4. 稳健性 | Robustness | 对参数、算法、样本不敏感 |
| 5. 可投资性 | Investability | 考虑交易成本后仍可获利 |
| 6. 普适性 | Pervasiveness | 在多个市场/时间段都有效 |

In [ ]:
# ========== 步骤 7: 模拟六大标准的评估 ==========
np.random.seed(42)

# 模拟 3 个候选预测变量
predictors = {
    'BM (账面市值比)': {'logic': 9, 'persistence': 8, 'increment': 7, 'robustness': 8, 'investability': 7, 'pervasiveness': 9},
    'IVOL (特质波动率)': {'logic': 7, 'persistence': 6, 'increment': 8, 'robustness': 5, 'investability': 6, 'pervasiveness': 7},
    'Mystery_Var': {'logic': 3, 'persistence': 7, 'increment': 5, 'robustness': 4, 'investability': 8, 'pervasiveness': 6},
}

criteria = ['logic', 'persistence', 'increment', 'robustness', 'investability', 'pervasiveness']
criteria_cn = ['逻辑性', '持续性', '信息增量', '稳健性', '可投资性', '普适性']

print("📊 预测变量六大标准评分（1-10 分）")
print(f"{'标准':<12}", end='')
for name in predictors:
    print(f"{name:<20}", end='')
print()
print("-" * 52)

for c, cn in zip(criteria, criteria_cn):
    print(f"{cn:<12}", end='')
    for name in predictors:
        print(f"{predictors[name][c]:<20}", end='')
    print()

print(f"\n总分{'':<8}", end='')
for name in predictors:
    total = sum(predictors[name].values())
    print(f"{total:<20}", end='')
print()

# 可视化雷达图
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

angles = np.linspace(0, 2 * np.pi, len(criteria), endpoint=False).tolist()
angles += angles[:1]

colors_radar = ['#2ecc71', '#e74c3c', '#3498db']
for idx, (name, scores) in enumerate(predictors.items()):
    values = [scores[c] for c in criteria]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=name, color=colors_radar[idx])
    ax.fill(angles, values, alpha=0.1, color=colors_radar[idx])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(criteria_cn, fontsize=10)
ax.set_ylim(0, 10)
ax.set_title('Predictor Quality Radar Chart', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0), fontsize=10)
plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  雷达图展示三个候选预测变量在六大标准上的表现。")
print(f"  BM（绿色）在逻辑性和普适性上得分最高，是经典预测变量。")
print(f"  Mystery_Var（蓝色）逻辑性得分低，可能存在数据窥探风险。")

---

## 8. 信息衰减与半衰期

### 📐 信息衰减

预测变量的信息含量会随时间衰减。Grinold and Kahn (1999) 定义了**半衰期**指标：

计算 $\text{corr}(z_{i,t}, R_{i,t+s+1})$ 随滞后期数 $s$ 的变化，
当 IC 下降到原始 IC 一半时的期数即为半衰期。

- 基于高频量价数据的变量：衰减快，半衰期短
- 基于财务数据的变量：衰减慢，半衰期长

In [ ]:
# ========== 步骤 8: 信息衰减模拟 ==========
N_STOCKS = 300
N_LAGS = 20  # 最大滞后期数

# 模拟两种变量的信息衰减
np.random.seed(42)

# 生成一个有自相关的预测变量
z_base = np.random.normal(0, 1, N_STOCKS)

ic_decay_fast = []  # 高频变量（衰减快）
ic_decay_slow = []  # 基本面变量（衰减慢）

for lag in range(N_LAGS):
    # 高频变量：IC 按指数衰减
    noise_fast = np.random.normal(0, 1, N_STOCKS)
    R_fast = z_base * np.exp(-0.3 * lag) + noise_fast
    ic_fast, _ = stats.pearsonr(z_base, R_fast)
    ic_decay_fast.append(abs(ic_fast))
    
    # 基本面变量：IC 衰减慢
    noise_slow = np.random.normal(0, 1, N_STOCKS)
    R_slow = z_base * np.exp(-0.08 * lag) + noise_slow
    ic_slow, _ = stats.pearsonr(z_base, R_slow)
    ic_decay_slow.append(abs(ic_slow))

# 找半衰期
ic_0_fast = ic_decay_fast[0]
half_fast = ic_0_fast / 2
half_life_fast = next((i for i, v in enumerate(ic_decay_fast) if v < half_fast), N_LAGS)

ic_0_slow = ic_decay_slow[0]
half_slow = ic_0_slow / 2
half_life_slow = next((i for i, v in enumerate(ic_decay_slow) if v < half_slow), N_LAGS)

print("📊 信息衰减分析")
print(f"  高频变量初始 IC  = {ic_0_fast:.4f}")
print(f"  高频变量半衰期  = {half_life_fast} 期")
print(f"  基本面变量初始 IC = {ic_0_slow:.4f}")
print(f"  基本面变量半衰期 = {half_life_slow} 期")

In [ ]:
# ========== 可视化信息衰减 ==========
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(range(N_LAGS), ic_decay_fast, 'o-', color='#e74c3c', linewidth=2,
        label=f'High-Freq (half-life = {half_life_fast} periods)')
ax.plot(range(N_LAGS), ic_decay_slow, 's-', color='#2ecc71', linewidth=2,
        label=f'Fundamental (half-life = {half_life_slow} periods)')

ax.axhline(y=ic_0_fast/2, color='#e74c3c', linestyle=':', alpha=0.5)
ax.axhline(y=ic_0_slow/2, color='#2ecc71', linestyle=':', alpha=0.5)
ax.axvline(x=half_life_fast, color='#e74c3c', linestyle=':', alpha=0.5)
ax.axvline(x=half_life_slow, color='#2ecc71', linestyle=':', alpha=0.5)

ax.set_xlabel('Lag (periods)', fontsize=12)
ax.set_ylabel('|IC|', fontsize=12)
ax.set_title('Information Decay: High-Freq vs Fundamental Predictors', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  红色（高频变量）：IC 衰减快，半衰期短，需要高换手率。")
print(f"  绿色（基本面变量）：IC 衰减慢，半衰期长，适合低频调仓。")

---

## 9. α 模型与风险模型的配合

### 📖 书中的要点

在实际投资中，α 模型和风险模型是因子投资的两大支柱：

| 模块 | 功能 | 输入 | 输出 |
|------|------|------|------|
| α 模型 | 预测收益率 | 预测变量 z | 预期收益率 $\hat{R}$ |
| 风险模型 | 预测风险 | 因子暴露 β | 协方差矩阵 $\Sigma$ |
| 组合优化 | 构建组合 | $\hat{R}$, $\Sigma$ | 最优权重 $w$ |

```
预测变量 z_it     因子暴露 β_it
      ↓                   ↓
  α 模型              风险模型
  预测 R̂_it          估计 Σ
      ↓                   ↓
      └──→ 组合优化 ←──┘
             ↓
         最优权重 w
```

> α 模型告诉你"买什么能赚钱"，风险模型告诉你"怎样控制风险"。
> 两者缺一不可：只看 α 不看风险可能导致集中度过高，只看风险不看 α 则无法获得超额收益。

In [ ]:
# ========== 步骤 9: 简化示例 - α 模型 + 风险模型 ==========
N_STOCKS = 10
np.random.seed(42)

stock_names = [f'S_{i+1}' for i in range(N_STOCKS)]

# α 模型：预测收益率
alpha_hat = np.array([0.5, 0.3, 0.8, -0.2, 0.6, 0.1, -0.4, 0.7, 0.2, -0.1])

# 风险模型：简化协方差矩阵（对角阵 + 小相关性）
vol = np.array([2.0, 1.5, 2.5, 1.8, 2.2, 1.6, 2.8, 1.9, 2.1, 1.7])  # 波动率
corr_matrix = np.eye(N_STOCKS)
for i in range(N_STOCKS):
    for j in range(N_STOCKS):
        if i != j:
            corr_matrix[i, j] = 0.3  # 简化：所有股票间相关性 = 0.3

D = np.diag(vol)
cov_matrix = D @ corr_matrix @ D  # 协方差矩阵

# 简化组合优化：均值-方差
# max w'α - λ/2 × w'Σw, s.t. sum(w) = 1
risk_aversion = 2.0

# 解析解（带约束的均值-方差）
ones = np.ones(N_STOCKS)
cov_inv = np.linalg.inv(cov_matrix)
A = ones @ cov_inv @ ones
B = ones @ cov_inv @ alpha_hat
C = alpha_hat @ cov_inv @ alpha_hat

w_mvo = cov_inv @ (alpha_hat - (B / A) * ones) / risk_aversion + (1 / A) * cov_inv @ ones

print("📊 α 模型 + 风险模型 → 组合优化")
print(f"{'股票':<8} {'预测收益 α':<15} {'波动率':<12} {'最优权重':<12}")
print("-" * 47)
for i in range(N_STOCKS):
    print(f"{stock_names[i]:<8} {alpha_hat[i]:<15.2f} {vol[i]:<12.2f} {w_mvo[i]:<12.4f}")

print(f"\n  权重之和 = {w_mvo.sum():.4f}")
print(f"  组合预期收益 = {w_mvo @ alpha_hat:.4f}")
print(f"  组合波动率   = {np.sqrt(w_mvo @ cov_matrix @ w_mvo):.4f}")
print(f"\n  💡 解读: 高 α + 低波动率的股票获得更高权重，体现了 α-风险平衡。")

---

## 📌 核心概念回顾

### 📌 收益率模型 (Alpha Model)
- **定义**: 预测股票未来收益率的模型，目标是获得超额收益
- **核心**: 寻找有效的收益预测变量 $z_{it}$
- **方法**: IC 检验、Fama-MacBeth 回归、投资组合排序法

### 📌 信息系数 IC
- **定义**: $IC = \text{corr}(z_{i,t}, R_{i,t+1})$
- **含义**: 衡量预测变量的单期预测能力
- **判断标准**: IC > 2% 为优秀，IR = IC_mean / IC_std > 0.5 为良好

### 📌 Fama-MacBeth 回归
- **定义**: 每期做截面回归，得到因子收益率时间序列
- **公式**: $R_{i,t} = \lambda_{0,t} + \lambda_{1,t} \cdot z_{i,t-1} + \varepsilon_{i,t}$
- **检验**: 对 $\lambda_1$ 做 T 检验，判断因子是否显著

### 📌 六大筛选标准
- **逻辑性**: 有经济学解释（风险补偿或错误定价）
- **持续性**: 实证上能持续预测收益
- **信息增量性**: 相对于已有变量有增量信息
- **稳健性**: 对参数、算法、样本不敏感
- **可投资性**: 考虑交易成本后仍可获利
- **普适性**: 在多个市场/时间段都有效

### 🔗 完整流程
```
寻找预测变量 → IC 检验 → Fama-MacBeth 回归 → 排序法验证
      ↓              ↓              ↓              ↓
  六大标准筛选    单期预测力     因子收益率      多空组合
      ↓              ↓              ↓              ↓
  信息衰减分析    IR 评估       T 检验          单调性
```

### 📝 检验指标汇总

| 指标 | 公式 | 含义 | 经验阈值 |
|------|------|------|----------|
| IC | $\text{corr}(z_t, R_{t+1})$ | 单期预测力 | > 2% |
| IR | $IC_{mean} / IC_{std}$ | 预测稳定性 | > 0.5 |
| FM t-stat | $\lambda_{mean} / SE(\lambda)$ | 因子显著性 | > 2.0 |
| Spread t-stat | $\mu_s / SE(s)$ | 多空收益显著性 | > 2.0 |
| Spearman $\rho$ | 秩相关 | 分组单调性 | > 0.8 |
| Half-life | IC 衰减到 50% 的期数 | 信息有效期 | 因变量而异 |

## ❌ 常见误区

### ❌ 误区 1: "阿尔法因子"就是因子
**✓ 正确理解**: 本书中"阿尔法"和"因子"指的是投资组合（如价值因子、特质波动率异象），而非背后的变量。业界说的"阿尔法因子"应称为"收益预测变量"。

### ❌ 误区 2: IC 高就一定有效
**✓ 正确理解**: IC 高只说明单期预测能力强，还需检验 IC 的持续性（IR）、统计显著性（t-value）以及信息衰减速度。一个 IC=5% 但 IR=0.2 的变量，不如 IC=3% 但 IR=0.8 的变量。

### ❌ 误区 3: 预测变量越多越好
**✓ 正确理解**: 多重假设检验会导致虚假发现。每个新变量必须通过信息增量性检验（控制已有变量后仍有预测力），否则只是已有变量的"影子"。

### ❌ 误区 4: 样本内有效 = 样本外有效
**✓ 正确理解**: 预测变量可能存在数据窥探（data snooping）风险。必须做发表前后检验、参数稳健性检验，以及在不同市场/时间段验证普适性。

### ❌ 误区 5: α 模型可以独立于风险模型使用
**✓ 正确理解**: α 模型只告诉你"买什么能赚钱"，但不考虑风险。实际投资中必须配合风险模型（如 Barra）进行组合优化，平衡收益和风险。